# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VipS-2004/flyrank1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!git clone https://github.com/VipS-2004/flyrank1.git

fatal: destination path 'flyrank1' already exists and is not an empty directory.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I created a simple baseline rule to identify content pages that should be reviewed first for content refresh.

The rule gives a higher score to pages that are older, have lower click-through rates (CTR), and lower engagement rates. These signals are available before making the decision and do not rely on future information.

### Reason Code
- STALE_CONTENT – The page is old and likely needs updating.
- LOW_CTR – The page has a low click-through rate.
- LOW_ENGAGEMENT – Users show low engagement with the content.

### Action

Refresh Content

This baseline rule provides a transparent ranking that can later be compared against a machine learning model.

In [7]:
import pandas as pd

df = pd.read_csv("/content/flyrank1/data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

print("\nAverage Content Age:")
print(df["content_age_days"].describe())

print("\nAverage CTR:")
print(df["ctr"].describe())

print("\nAverage Engagement Rate:")
print(df["engagement_rate"].describe())


Dataset Shape: (30000, 44)

Average Content Age:
count    30000.00000
mean       256.16780
std        132.70793
min         90.00000
25%        132.00000
50%        236.00000
75%        333.00000
max        564.00000
Name: content_age_days, dtype: float64

Average CTR:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

Average Engagement Rate:
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import os

# Baseline Score
df["baseline_score"] = (
    0.4 * (df["content_age_days"] / df["content_age_days"].max())
    +
    0.3 * (1 - df["ctr"] / 100)
    +
    0.3 * (1 - df["engagement_rate"] / 100)
)

# Reason Code
def reason(row):
    if row["content_age_days"] > 365:
        return "STALE_CONTENT"
    elif row["ctr"] < 2:
        return "LOW_CTR"
    else:
        return "LOW_ENGAGEMENT"

df["reason_code"] = df.apply(reason, axis=1)

# Action
df["action"] = "Refresh Content"

# Rank
queue = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("/content/flyrank1/work/outputs", exist_ok=True)

queue.to_csv(
    "/content/flyrank1/work/outputs/baseline_action_score.csv",
    index=False
)

print(queue[["content_id","baseline_score","reason_code","action"]].head(20))

                 content_id  baseline_score    reason_code           action
3604   content_5f7d77cf01e3        0.995035  STALE_CONTENT  Refresh Content
2115   content_a0c3be8b0794        0.995035  STALE_CONTENT  Refresh Content
7500   content_5b5e85993c2b        0.995035  STALE_CONTENT  Refresh Content
19810  content_9d17befb32b0        0.995035  STALE_CONTENT  Refresh Content
7397   content_bb5a87d3be4f        0.995035  STALE_CONTENT  Refresh Content
25831  content_5466a8258699        0.995035  STALE_CONTENT  Refresh Content
25956  content_7faeb2d774be        0.995035  STALE_CONTENT  Refresh Content
12934  content_70641aa29f3e        0.995035  STALE_CONTENT  Refresh Content
12972  content_d5b833d82e72        0.995035  STALE_CONTENT  Refresh Content
17338  content_8e4c8d698268        0.995035  STALE_CONTENT  Refresh Content
15292  content_2f116a1471f2        0.995035  STALE_CONTENT  Refresh Content
8990   content_a38c8f61e246        0.995035  STALE_CONTENT  Refresh Content
4565   conte

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive high scores simply because they are old, even if they remain useful and accurate. These pages should be manually reviewed before taking action.

## Leakage Check

No future performance metrics or label-derived columns were used.

The baseline rule only uses historical information that would already be available when making the refresh decision:

- Content Age
- CTR
- Engagement Rate

Therefore, no data leakage is present.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.